LABORATORIO 4 - Modelo vectorial

Representa los documentos de la coleccion CACM y las consultas en el modelo vectorial,
usando el vocabulario reducido (ordenado alfabeticamente) de la practica 3.

Entradas (practica 2): documentos_preprocesados.txt y consultas_preprocesadas.txt (formato: Numero | titulo | texto).

Entrada (practica 3): vocabularioReducido.txt, ordenado alfabeticamente (columnas de las matrices).

Salidas (carpeta lab4):
- matriz_tf_documentos / matriz_tf_consultas: esquema de pesado tf (frecuencia del termino)
- matriz_tfidf_documentos / matriz_tfidf_consultas: esquema de pesado tf-idf, idf = log10(N / df)

Cada matriz se guarda en .npz (formato disperso de scipy) para usarla en practicas posteriores.

In [1]:
import os

In [2]:
import math

In [3]:
from collections import Counter

In [4]:
import numpy as np

In [5]:
from scipy import sparse

In [6]:
#CARPETAS
carpeta_entrada = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica2"
carpeta_vocabulario = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica3"
corpus_root = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica4"

LECTURA DE LOS DOCUMENTOS Y CONSULTAS PREPROCESADOS

In [7]:
# cada linea es "Numero | titulo | texto" se juntan titulo y texto como las palabras del documento
def leer_preprocesado(nombre_archivo):
    with open(os.path.join(carpeta_entrada, nombre_archivo), encoding="utf-8") as f:
        lineas = f.read().splitlines()
    registros = {}
    for linea in lineas:
        num, titulo, texto = linea.split(" | ", 2)
        registros[num.strip()] = (titulo + " " + texto).split()
    return registros

In [8]:
documentos = leer_preprocesado("documentos_preprocesados.txt")
consultas = leer_preprocesado("consultas_preprocesadas.txt")
N = len(documentos)
print(N, "documentos leidos")
print(len(consultas), "consultas leidas")

3204 documentos leidos
64 consultas leidas


In [9]:
print(documentos["20"])  
print(consultas["1"])    

['acceler', 'converg', 'iter', 'process', 'techniqu', 'discuss', 'appli', 'iter', 'procedur', 'solut', 'equat', 'acceler', 'rate', 'converg', 'iter', 'converg', 'induc', 'converg', 'iter', 'diverg', 'illustr', 'exampl', 'given']
['articl', 'exist', 'deal', 'tss', 'time', 'share', 'system', 'oper', 'system', 'ibm', 'comput']


VOCABULARIO REDUCIDO 

Se lee vocabularioReducido.txt y se ordena alfabeticamente por si acaso.

In [10]:
with open(os.path.join(carpeta_vocabulario, "vocabularioReducido.txt"), encoding="utf-8") as f:
    vocabulario_reducido = sorted(f.read().split())   # ordenado alfabeticamente
print("Longitud del vocabulario reducido:", len(vocabulario_reducido))
print("Primeros 30 terminos:", vocabulario_reducido[:30])

Longitud del vocabulario reducido: 3016
Primeros 30 terminos: ['a', 'ab', 'abandon', 'abbrevi', 'abil', 'abl', 'abscissa', 'absenc', 'absolut', 'abstract', 'academ', 'acceler', 'accept', 'access', 'accommod', 'accompani', 'accomplish', 'accord', 'account', 'accru', 'accumul', 'accur', 'accuraci', 'achiev', 'acm', 'acquir', 'acquisit', 'across', 'act', 'action']


In [11]:
# posicion (columna) de cada termino dentro del vector
indice = {termino: j for j, termino in enumerate(vocabulario_reducido)}
M = len(vocabulario_reducido)
print("Dimension de los vectores:", M)

Dimension de los vectores: 3016


ESQUEMA DE PESADO TF

Cada documento (o consulta) es un vector de dimension M = tamaño del vocabulario reducido.
La entrada j es el numero de veces que el termino j aparece en el documento.
Las palabras que no estan en el vocabulario reducido se ignoran.

In [12]:
def matriz_tf(registros):
    ids = list(registros)
    tf = np.zeros((len(ids), M), dtype=np.int32)
    for i, num in enumerate(ids):
        for palabra in registros[num]:
            if palabra in indice:
                tf[i, indice[palabra]] += 1
    return ids, tf

In [13]:
ids_docs, tf_docs = matriz_tf(documentos)
ids_consultas, tf_consultas = matriz_tf(consultas)
print("Matriz tf de documentos:", tf_docs.shape)
print("Matriz tf de consultas:", tf_consultas.shape)

Matriz tf de documentos: (3204, 3016)
Matriz tf de consultas: (64, 3016)


ESQUEMA DE PESADO TF-IDF

w(i, j) = tf(i, j) * idf(j), con idf(j) = log10(N / df(j)).
El idf se calcula con la coleccion de documentos y se usa el mismo para las consultas.

In [14]:
# numero de documentos donde aparece cada termino (df)
df = Counter()
for palabras in documentos.values():
    df.update(set(palabras))   # set: cada documento cuenta una sola vez

In [15]:
idf = np.array([math.log10(N / df[w]) for w in vocabulario_reducido])
print("idf minimo: %.4f (%s)" % (idf.min(), vocabulario_reducido[idf.argmin()]))
print("idf maximo: %.4f (%s)" % (idf.max(), vocabulario_reducido[idf.argmax()]))

idf minimo: 0.7914 (present)
idf maximo: 3.2047 (a)


In [16]:
tfidf_docs = tf_docs * idf
tfidf_consultas = tf_consultas * idf
print("Matriz tf-idf de documentos:", tfidf_docs.shape)
print("Matriz tf-idf de consultas:", tfidf_consultas.shape)

Matriz tf-idf de documentos: (3204, 3016)
Matriz tf-idf de consultas: (64, 3016)


GUARDAR LAS MATRICES

In [17]:
def guardar_matriz(matriz, ids, nombre):
    # .npz disperso, para cargarla en practicas posteriores con sparse.load_npz
    # renglon = documento o consulta (orden en ids_*.txt), columna = termino (orden en vocabularioReducido.txt de la practica 3)
    sparse.save_npz(os.path.join(corpus_root, nombre + ".npz"), sparse.csr_matrix(matriz))
    print(matriz.shape, "->", nombre + ".npz")

In [18]:
guardar_matriz(tf_docs, ids_docs, "matriz_tf_documentos")
guardar_matriz(tf_consultas, ids_consultas, "matriz_tf_consultas")
guardar_matriz(tfidf_docs, ids_docs, "matriz_tfidf_documentos")
guardar_matriz(tfidf_consultas, ids_consultas, "matriz_tfidf_consultas")

(3204, 3016) -> matriz_tf_documentos.npz
(64, 3016) -> matriz_tf_consultas.npz
(3204, 3016) -> matriz_tfidf_documentos.npz
(64, 3016) -> matriz_tfidf_consultas.npz


In [19]:
# orden de los renglones de las matrices (numero de documento / consulta)
with open(os.path.join(corpus_root, "ids_documentos.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(ids_docs))
with open(os.path.join(corpus_root, "ids_consultas.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(ids_consultas))

VERIFICACION (para la seccion de resultados del reporte)

Se elige un termino del vocabulario reducido y un documento preprocesado donde aparece
mas de una vez, y se muestra el renglon correspondiente de la matriz tf.

In [20]:
termino = "converg"
num_doc = "20"

In [21]:
# posicion del termino en el vocabulario ordenado alfabeticamente
j = indice[termino]
print("Termino:", termino, "-> columna", j, "del vocabulario reducido")
print("Vecinos en el vocabulario:", vocabulario_reducido[j - 3:j + 4])

Termino: converg -> columna 574 del vocabulario reducido
Vecinos en el vocabulario: ['control', 'conveni', 'convent', 'converg', 'convers', 'convert', 'convex']


In [22]:
# documento preprocesado 
with open(os.path.join(carpeta_entrada, "documentos_preprocesados.txt"), encoding="utf-8") as f:
    for linea in f:
        if linea.startswith(num_doc + " | "):
            print(linea.strip())
print("Veces que aparece '%s':" % termino, documentos[num_doc].count(termino))

20 | acceler converg iter process | techniqu discuss appli iter procedur solut equat acceler rate converg iter converg induc converg iter diverg illustr exampl given
Veces que aparece 'converg': 4


In [23]:
# renglon del documento en la matriz tf (solo las entradas distintas de cero)
i = ids_docs.index(num_doc)
renglon = tf_docs[i]
print("Documento", num_doc, "-> renglon", i, "de la matriz tf, dimension", len(renglon))
for k in np.nonzero(renglon)[0]:
    print("  %5d  %-12s tf = %d" % (k, vocabulario_reducido[k], renglon[k]))
print("tf['%s'] en el documento %s = %d" % (termino, num_doc, tf_docs[i, j]))

Documento 20 -> renglon 19 de la matriz tf, dimension 3016
     11  acceler      tf = 2
    131  appli        tf = 1
    574  converg      tf = 4
    765  discuss      tf = 1
    778  diverg       tf = 1
    905  equat        tf = 1
    941  exampl       tf = 1
   1130  given        tf = 1
   1270  illustr      tf = 1
   1320  induc        tf = 1
   1420  iter         tf = 4
   2082  procedur     tf = 1
   2084  process      tf = 1
   2177  rate         tf = 1
   2500  solut        tf = 1
   2691  techniqu     tf = 1
tf['converg'] en el documento 20 = 4


In [24]:
# el mismo renglon leido desde el archivo guardado
tf_guardada = sparse.load_npz(os.path.join(corpus_root, "matriz_tf_documentos.npz")).toarray()
fila = tf_guardada[i]
for k in np.nonzero(fila)[0]:
    print("  %5d  %-12s tf = %d" % (k, vocabulario_reducido[k], fila[k]))

     11  acceler      tf = 2
    131  appli        tf = 1
    574  converg      tf = 4
    765  discuss      tf = 1
    778  diverg       tf = 1
    905  equat        tf = 1
    941  exampl       tf = 1
   1130  given        tf = 1
   1270  illustr      tf = 1
   1320  induc        tf = 1
   1420  iter         tf = 4
   2082  procedur     tf = 1
   2084  process      tf = 1
   2177  rate         tf = 1
   2500  solut        tf = 1
   2691  techniqu     tf = 1


In [25]:
# renglon del documento en la matriz tf-idf
renglon = tfidf_docs[i]
for k in np.nonzero(renglon)[0]:
    print("  %5d  %-12s tf = %d  idf = %.4f  tf-idf = %.4f" % (k, vocabulario_reducido[k], tf_docs[i, k], idf[k], renglon[k]))

     11  acceler      tf = 2  idf = 2.7275  tf-idf = 5.4551
    131  appli        tf = 1  idf = 1.4193  tf-idf = 1.4193
    574  converg      tf = 4  idf = 2.0143  tf-idf = 8.0573
    765  discuss      tf = 1  idf = 0.9542  tf-idf = 0.9542
    778  diverg       tf = 1  idf = 3.0286  tf-idf = 3.0286
    905  equat        tf = 1  idf = 1.3504  tf-idf = 1.3504
    941  exampl       tf = 1  idf = 1.2112  tf-idf = 1.2112
   1130  given        tf = 1  idf = 0.9494  tf-idf = 0.9494
   1270  illustr      tf = 1  idf = 1.5101  tf-idf = 1.5101
   1320  induc        tf = 1  idf = 2.5514  tf-idf = 2.5514
   1420  iter         tf = 4  idf = 1.7575  tf-idf = 7.0300
   2082  procedur     tf = 1  idf = 1.1459  tf-idf = 1.1459
   2084  process      tf = 1  idf = 0.9446  tf-idf = 0.9446
   2177  rate         tf = 1  idf = 1.9146  tf-idf = 1.9146
   2500  solut        tf = 1  idf = 1.2003  tf-idf = 1.2003
   2691  techniqu     tf = 1  idf = 0.9755  tf-idf = 0.9755


In [26]:
# ejemplo de una consulta representada con tf y tf-idf
num_q = "1"
q = ids_consultas.index(num_q)
print("Consulta", num_q, ":", " ".join(consultas[num_q]))
for k in np.nonzero(tf_consultas[q])[0]:
    print("  %5d  %-12s tf = %d  tf-idf = %.4f" % (k, vocabulario_reducido[k], tf_consultas[q, k], tfidf_consultas[q, k]))

Consulta 1 : articl exist deal tss time share system oper system ibm comput
    162  articl       tf = 1  tf-idf = 2.2047
    664  deal         tf = 1  tf-idf = 1.9375
    954  exist        tf = 1  tf-idf = 1.5514
   1252  ibm          tf = 1  tf-idf = 1.5280
   1842  oper         tf = 1  tf-idf = 0.9859
   2425  share        tf = 1  tf-idf = 1.5145
   2748  time         tf = 1  tf-idf = 0.8908
